In [13]:
# Load modular data 

import gurobipy as gp
from gurobipy import GRB
import pandas as pd

m = gp.Model("MIP")

from ranDataGen.order_data_loader import load_order_data

# Load from ranDataGen copy; change size and sample to switch datasets
size = "large"
sample = 1

base_dir = f"ranDataGen copy/{size} sized samples/"

itemtypes_path = f"order_itemtypes_{sample}.csv"
quantities_path = f"order_quantities_{sample}.csv"
totes_path = f"orders_totes_{sample}.csv"

# Load baseline data using modular loader
# df = load_order_data(
#     itemtypes_path=itemtypes_path,
#     quantities_path=quantities_path,
#     totes_path=totes_path,
#     base_dir=base_dir,
# )

df = load_order_data(
    itemtypes_path="order_itemtypes.csv",
    quantities_path="order_quantities.csv",
    totes_path="orders_totes.csv",
    base_dir="ranDataGen/",
)

df

,order,item_slot,itemtype,quantity,tote
0,1,0,3,3,1
1,1,1,1,2,1
2,2,0,2,3,2
3,2,1,3,1,3
4,2,2,0,1,2
5,3,0,3,3,3
6,3,1,2,3,2
7,3,2,0,1,1
8,4,0,1,1,0
9,4,1,2,1,0


# Time Based Model

In [20]:
# ==================================================
# Greedy cascade model with upstream belt priority
# ==================================================

import gurobipy as gp
from gurobipy import GRB

m_cascade = gp.Model("MIP_greedy_cascade")

# Use same data frame df already loaded above

# --------------------------------------------------
# Build fixed item sequence along conveyor
# HARD-CODED ITEM POSITIONS for greedy cascade MIP
# --------------------------------------------------

# items_seq is a list of (item_id, item_type, tote)
# Positions 1..19 are fixed as specified:
# Position 1:  Item 1  | Type 3 | Tote 1
# Position 2:  Item 2  | Type 3 | Tote 1
# Position 3:  Item 3  | Type 3 | Tote 1
# Position 4:  Item 4  | Type 1 | Tote 1
# Position 5:  Item 5  | Type 1 | Tote 1
# Position 6:  Item 17 | Type 0 | Tote 1
# Position 7:  Item 6  | Type 2 | Tote 2
# Position 8:  Item 7  | Type 2 | Tote 2
# Position 9:  Item 8  | Type 2 | Tote 2
# Position 10: Item 10 | Type 0 | Tote 2
# Position 11: Item 14 | Type 2 | Tote 2
# Position 12: Item 15 | Type 2 | Tote 2
# Position 13: Item 16 | Type 2 | Tote 2
# Position 14: Item 9  | Type 3 | Tote 3
# Position 15: Item 11 | Type 3 | Tote 3
# Position 16: Item 12 | Type 3 | Tote 3
# Position 17: Item 13 | Type 3 | Tote 3
# Position 18: Item 18 | Type 1 | Tote 0
# Position 19: Item 19 | Type 2 | Tote 0

items_seq = [
    (1, 3, 1),   # 1
    (2, 3, 1),   # 2
    (3, 3, 1),   # 3
    (4, 1, 1),   # 4
    (5, 1, 1),   # 5
    (17, 0, 1),  # 6
    (6, 2, 2),   # 7
    (7, 2, 2),   # 8
    (8, 2, 2),   # 9
    (10, 0, 2),  # 10
    (14, 2, 2),  # 11
    (15, 2, 2),  # 12
    (16, 2, 2),  # 13
    (9, 3, 3),   # 14
    (11, 3, 3),  # 15
    (12, 3, 3),  # 16
    (13, 3, 3),  # 17
    (18, 1, 0),  # 18
    (19, 2, 0),  # 19
]

N = len(items_seq)
positions = range(1, N + 1)

# Maps from item id to position and type
pos_of_iid = {}
type_of_iid = {}
iid_at_pos = {}

for p, (iid, itype, tote) in enumerate(items_seq, start=1):
    pos_of_iid[iid] = p
    type_of_iid[iid] = itype
    iid_at_pos[p] = iid

# --------------------------------------------------
# Sets and demand
# --------------------------------------------------

belts = range(1, 5)
orders = list(df["order"].unique())
item_types = sorted(df["itemtype"].unique())

# Demand per order and item type
Demand = {}
for _, row in df.iterrows():
    o = int(row["order"])
    itype = int(row["itemtype"])
    qty = int(row["quantity"])
    if qty <= 0:
        continue
    Demand[(o, itype)] = Demand.get((o, itype), 0) + qty

# Precompute max demand per (order, type)
max_dem = {(o, t): Demand.get((o, t), 0) for o in orders for t in item_types}

# --------------------------------------------------
# Decision variables
# --------------------------------------------------

# Assign each order to exactly one belt
y_c = m_cascade.addVars(
    [(b, o) for b in belts for o in orders],
    vtype=GRB.BINARY,
    name="assign_c",
)

m_cascade.addConstrs(
    (gp.quicksum(y_c[b, o] for b in belts) == 1 for o in orders),
    name="assign_one_belt_c",
)

# Pick variables: belt b uses physical item iid for order o
w = m_cascade.addVars(
    [(b, iid, o) for b in belts for (iid, _, _) in items_seq for o in orders],
    vtype=GRB.BINARY,
    name="pick_c",
)

# Each item used at most once
m_cascade.addConstrs(
    (
        gp.quicksum(w[b, iid, o] for b in belts for o in orders) <= 1
        for (iid, _, _) in items_seq
    ),
    name="use_item_once_c",
)

# Picks only allowed on the belt the order is assigned to
m_cascade.addConstrs(
    (w[b, iid, o] <= y_c[b, o] for b in belts for (iid, _, _) in items_seq for o in orders),
    name="pick_implies_assign_c",
)

# Remove invalid picks where order has no demand for that item type
valid_pairs = set(Demand.keys())  # (order, itype)
for (iid, itype, _) in items_seq:
    for o in orders:
        if (o, itype) not in valid_pairs:
            for b in belts:
                w[b, iid, o].ub = 0

# Demand satisfaction: for each (order, type), sum of picked items equals quantity
for (o, itype), qty in Demand.items():
    m_cascade.addConstr(
        gp.quicksum(
            w[b, iid, o]
            for b in belts
            for (iid, it, _) in items_seq
            if it == itype
        )
        == qty,
        name=f"demand_c_o{o}_t{itype}",
    )

# --------------------------------------------------
# Order sequencing on each belt: no interleaving of orders
# A belt can handle multiple orders, but must finish one
# order before starting to pick for the next.
# --------------------------------------------------

# First and last pick positions for each (belt, order)
f_pos = m_cascade.addVars(
    [(b, o) for b in belts for o in orders],
    vtype=GRB.CONTINUOUS,
    lb=1.0,
    ub=N,
    name="first_pos",
)

l_pos = m_cascade.addVars(
    [(b, o) for b in belts for o in orders],
    vtype=GRB.CONTINUOUS,
    lb=1.0,
    ub=N,
    name="last_pos",
)

BigN = float(N)

# If belt b picks order o at position k, then that position
# must lie inside [f_pos, l_pos] for that (b,o).
for b in belts:
    for o in orders:
        for k in range(1, N + 1):
            iid_k = iid_at_pos[k]
            # When w[b,iid_k,o] = 1, these reduce to
            #   f_pos[b,o] <= k
            #   l_pos[b,o] >= k
            m_cascade.addConstr(
                f_pos[b, o] <= k + BigN * (1 - w[b, iid_k, o]),
                name=f"firstpos_ub_b{b}_o{o}_k{k}",
            )
            m_cascade.addConstr(
                l_pos[b, o] >= k - BigN * (1 - w[b, iid_k, o]),
                name=f"lastpos_lb_b{b}_o{o}_k{k}",
            )

# Pairwise order precedence on each belt: for any two
# orders assigned to the same belt, their pick intervals
# [f_pos, l_pos] must not overlap.

order_pairs = [
    (b, o1, o2)
    for b in belts
    for i, o1 in enumerate(orders)
    for o2 in orders[i + 1 :]
]

before = m_cascade.addVars(order_pairs, vtype=GRB.BINARY, name="order_before")

for b, o1, o2 in order_pairs:
    # If both orders are assigned to belt b (y_c[b,o1] = y_c[b,o2] = 1), then
    # either (o1 before o2) or (o2 before o1), and their intervals cannot overlap.
    #
    # When y_c[b,o1] = y_c[b,o2] = 1, these become the standard disjunctive
    # constraints:
    #   l_pos[b,o1] <= f_pos[b,o2] + N * (1 - before[b,o1,o2])
    #   l_pos[b,o2] <= f_pos[b,o1] + N * (before[b,o1,o2])
    # so intervals [f_pos, l_pos] for the two orders do not interleave.
    m_cascade.addConstr(
        l_pos[b, o1]
        <= f_pos[b, o2]
        + BigN
        * (1 - before[b, o1, o2] + (1 - y_c[b, o1]) + (1 - y_c[b, o2])),
        name=f"no_overlap1_b{b}_o{o1}_o{o2}",
    )
    m_cascade.addConstr(
        l_pos[b, o2]
        <= f_pos[b, o1]
        + BigN
        * (before[b, o1, o2] + (1 - y_c[b, o1]) + (1 - y_c[b, o2])),
        name=f"no_overlap2_b{b}_o{o1}_o{o2}",
    )

# --------------------------------------------------
# Remaining-demand flow along the conveyor
# rem[b,o,t,k] = remaining demand of type t for order o on belt b
# after processing positions up to k (k = 0..N)
# --------------------------------------------------

rem = {}
has_rem = {}

for b in belts:
    for o in orders:
        for t in item_types:
            if max_dem[(o, t)] == 0:
                continue
            for k in range(0, N + 1):
                rem[b, o, t, k] = m_cascade.addVar(
                    vtype=GRB.INTEGER,
                    lb=0,
                    ub=max_dem[(o, t)],
                    name=f"rem_b{b}_o{o}_t{t}_k{k}",
                )
                if k < N + 1:
                    has_rem[b, o, t, k] = m_cascade.addVar(
                        vtype=GRB.BINARY,
                        name=f"hasrem_b{b}_o{o}_t{t}_k{k}",
                    )

m_cascade.update()

# Initial remaining demand at k = 0: equals demand[o,t] if order o is assigned to belt b, else 0
for b in belts:
    for o in orders:
        for t in item_types:
            if max_dem[(o, t)] == 0:
                continue
            d_ot = max_dem[(o, t)]
            # If y_c[b,o] = 0 -> rem = 0; if y_c[b,o] = 1 -> rem = d_ot
            m_cascade.addConstr(
                rem[b, o, t, 0] <= d_ot * y_c[b, o],
                name=f"rem0_ub_b{b}_o{o}_t{t}",
            )
            m_cascade.addConstr(
                rem[b, o, t, 0] >= d_ot * y_c[b, o] - d_ot * (1 - y_c[b, o]),
                name=f"rem0_lb_b{b}_o{o}_t{t}",
            )

# Flow over positions k = 1..N
for b in belts:
    for o in orders:
        for t in item_types:
            if max_dem[(o, t)] == 0:
                continue
            d_ot = max_dem[(o, t)]
            for k in range(1, N + 1):
                iid_k = iid_at_pos[k]
                itype_k = type_of_iid[iid_k]
                pick_here = w[b, iid_k, o] if itype_k == t else 0
                # rem[b,o,t,k] = rem[b,o,t,k-1] - pick_here
                if itype_k == t:
                    m_cascade.addConstr(
                        rem[b, o, t, k]
                        == rem[b, o, t, k - 1] - w[b, iid_k, o],
                        name=f"rem_flow_b{b}_o{o}_t{t}_k{k}",
                    )
                else:
                    m_cascade.addConstr(
                        rem[b, o, t, k] == rem[b, o, t, k - 1],
                        name=f"rem_flow_b{b}_o{o}_t{t}_k{k}",
                    )
                # Link rem and has_rem at k-1 (whether there is still positive remaining demand)
                m_cascade.addConstr(
                    rem[b, o, t, k - 1]
                    <= d_ot * has_rem[b, o, t, k - 1],
                    name=f"hasrem_ub_b{b}_o{o}_t{t}_k{k-1}",
                )
                m_cascade.addConstr(
                    rem[b, o, t, k - 1]
                    >= has_rem[b, o, t, k - 1],
                    name=f"hasrem_lb_b{b}_o{o}_t{t}_k{k-1}",
                )

# --------------------------------------------------
# Greedy cascade: downstream belts cannot pick while
# any upstream belt still has remaining demand for that type
# at this position
# --------------------------------------------------

U = {}  # U[b,k] = 1 if any upstream belt (<b) still needs the type at position k

for b in belts:
    if b == 1:
        continue
    for k in range(1, N + 1):
        iid_k = iid_at_pos[k]
        t_k = type_of_iid[iid_k]
        if max_dem.get((orders[0], t_k), 0) is None:  # type safety; will be overwritten if used
            continue
        U[b, k] = m_cascade.addVar(
            vtype=GRB.BINARY, name=f"upstream_need_b{b}_k{k}"
        )

m_cascade.update()

for b in belts:
    if b == 1:
        continue
    for k in range(1, N + 1):
        iid_k = iid_at_pos[k]
        t_k = type_of_iid[iid_k]
        # If no order ever needs this type, skip
        if all(max_dem[(o, t_k)] == 0 for o in orders):
            continue
        # U[b,k] >= has_rem[b',o,t_k,k-1] for any upstream belt b'<b and order o
        for bp in belts:
            if bp >= b:
                continue
            for o in orders:
                if max_dem[(o, t_k)] == 0:
                    continue
                m_cascade.addConstr(
                    U[b, k] >= has_rem[bp, o, t_k, k - 1],
                    name=f"upstream_flag_b{b}_bp{bp}_o{o}_k{k}",
                )
        # If any upstream belt still needs type at k-1, belt b cannot pick this item
        m_cascade.addConstr(
            gp.quicksum(w[b, iid_k, o] for o in orders) <= 1 - U[b, k],
            name=f"no_downstream_pick_b{b}_k{k}",
        )

# --------------------------------------------------
# Time-based objective for cascade model
# --------------------------------------------------

TIME_c = m_cascade.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="completion_time_c")

Mtime = 3 * N + 7.5 * (max(belts) - 1) + 3

for b in belts:
    for (iid, _, _) in items_seq:
        p = pos_of_iid[iid]
        for o in orders:
            m_cascade.addConstr(
                TIME_c
                >= 3 * p + 7.5 * (b - 1) + 3 - Mtime * (1 - w[b, iid, o]),
                name=f"time_ge_pick_c_b{b}_i{iid}_o{o}",
            )

m_cascade.setObjective(TIME_c, GRB.MINIMIZE)

m_cascade.Params.TimeLimit = 60
m_cascade.Params.MIPGap = 0.05

m_cascade.optimize()

# ==================================================
# Print Results for greedy cascade model
# ==================================================

from pathlib import Path

out_dir = Path("MIP") / "output" / str(size)
out_dir.mkdir(parents=True, exist_ok=True)
output_txt_path = out_dir / f"model_output_cascade_{size}_{sample}.txt"

_output_lines = []

def log(s: str = ""):
    print(s)
    _output_lines.append(str(s))

log("\nITEM SEQUENCE ON CONVEYOR (fixed sequence)")

item_type_names = {
    0: "circle",
    1: "pentagon",
    2: "trapezoid",
    3: "triangle",
    4: "star",
    5: "moon",
    6: "heart",
    7: "cross",
}

sequence = []
for p in positions:
    iid, itype, tote = items_seq[p - 1]
    sequence.append((p, iid, itype, tote))

sequence.sort()
for p, iid, itype, tote in sequence:
    name = item_type_names.get(itype, f"type-{itype}")
    log(f"Position {p}: Item {iid} | Type {itype} ({name}) | Tote {tote}")

log("\nBELT → ORDER ASSIGNMENTS (cascade)")
for b in belts:
    for o in orders:
        if y_c[b, o].X > 0.5:
            log(f"Belt {b} processes Order {o}")

log("\nPICKS WITH TIME (cascade)")

picks_with_time = []
for b in belts:
    for (iid, itype, tote) in items_seq:
        for o in orders:
            if w[b, iid, o].X > 0.5:
                p = pos_of_iid[iid]
                t_pick = 3 * p + 7.5 * (b - 1) + 3
                picks_with_time.append((t_pick, b, iid, itype, tote, o))

picks_with_time.sort(key=lambda x: x[0])
for t_pick, b, iid, itype, tote, o in picks_with_time:
    log(
        f"Belt {b} picks Item {iid} (Type {itype}, Tote {tote}) for Order {o} → Time = {t_pick:.2f} sec"
    )

if m_cascade.SolCount > 0:
    log(f"\nSYSTEM COMPLETION TIME (cascade): {TIME_c.X}")

output_txt_path.write_text("\n".join(_output_lines) + "\n", encoding="utf-8")
log(f"\nCascade model output saved: {output_txt_path}")


Set parameter TimeLimit to value 60
Set parameter MIPGap to value 0.05
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  60
MIPGap  0.05

Optimize a model with 4026 rows, 2034 columns and 9037 nonzeros
Model fingerprint: 0x25d7fb8e
Variable types: 33 continuous, 2001 integer (1201 binary)
Coefficient statistics:
  Matrix range     [1e+00, 8e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e+00, 8e+01]
Presolve removed 3177 rows and 1661 columns
Presolve time: 0.41s
Presolved: 849 rows, 373 columns, 2750 nonzeros
Variable types: 0 continuous, 373 integer (312 binary)
Found heuristic solution: objective 82.5000000

Root relaxation: objective 4.860000e+01, 392 iterations, 0.04 seconds (0.01 work units)

    Nodes    |    Current Node    |     Obj

# CSV Input File

In [24]:
# Shape mapping
shape_map = {
    0: "circle",
    1: "pentagon",
    2: "trapezoid",
    3: "triangle",
    4: "star",
    5: "moon",
    6: "heart",
    7: "cross"
}

belts = range(1,5)

rows = []

for b in belts:
    row = {"conv_num": b}

    # Initialize columns
    for col in shape_map.values():
        row[col] = 0

    # Count picks by item type
    for item_id, item_type, tote in items:

        if item_type not in shape_map:
            continue

        shape_name = shape_map[item_type]

        count = 0

        for o in orders:
            if pick[b, item_id, o].X > 0.5:
                count += 1

        row[shape_name] += count

    rows.append(row)

# Create dataframe
df_picklist = pd.DataFrame(rows)

# Ensure correct column order
df_picklist = df_picklist[
    ["conv_num"] + list(shape_map.values())
]

# Save CSV to size/sample-specific folder
from pathlib import Path

out_dir = Path("MIP") / "output" / str(size)
out_dir.mkdir(parents=True, exist_ok=True)

output_path = out_dir / f"MIP-belt_picklist_{size}_{sample}.csv"
df_picklist.to_csv(output_path, index=False)

print("CSV generated:", output_path)

CSV generated: MIP/output/medium/MIP-belt_picklist_medium_3.csv
